<a href="https://colab.research.google.com/github/felondrum/llm_driven_development_otus/blob/develop/llm_otus_filippov_hw3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Шаг 1: Установка необходимых библиотек

    ┌───────────────────────┬───────────────────────────────────────────────────────┐
    │ Библиотека            │ Зачем                                                 │
    ├───────────────────────┼───────────────────────────────────────────────────────┤
    │ datasets              │ Загрузка датасета Sberquad из HuggingFace             │
    │ transformers + torch  │ Запуск локальных языковых моделей                     │
    │ evaluate, nltk        │ Расчёт BLEU-метрики                                   │
    │ sentence-transformers │ Semantic Similarity — косинусная близость эмбеддингов │
    │ pandas                │ Хранение и агрегация результатов                      │
    │ matplotlib, seaborn   │ Визуализация и сравнение моделей                      │

In [3]:
!pip install datasets transformers torch evaluate nltk sacrebleu sentence-transformers tqdm pandas matplotlib seaborn

Импорт зависимостей

    Импортируем всё необходимое для дальнейшей работы.
    
    ------
  
    Настраиваем seaborn для красивых график и скачиваем punkt для токенизации NLTK.

In [4]:
import os
import time
import re
import string
from collections import Counter

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from datasets import load_dataset

from transformers import (
  AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
)

import evaluate
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sentence_transformers import SentenceTransformer, util

import nltk
nltk.download('punkt', quiet=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

Загружаем датасет kuznetsoffandrey/sberquad — это русскоязычный QA-датасет, аналог SQuAD. Каждый пример содержит:

     - context — текст-контекст
     - question — вопрос по контексту
     - answers — эталонные ответы (список с позициями)

In [5]:
dataset = load_dataset("kuznetsoffandrey/sberquad")
print("Доступные сплиты:", dataset.keys())
print("\nСтруктура:")
print(dataset)

Доступные сплиты: dict_keys(['train', 'validation', 'test'])

Структура:
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 45328
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5036
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 23936
    })
})


Sberquad имеет вложенную структуру — answers это словарь с ключами text (список ответов) и answer_start (позиции). Убедимся, что правильно понимаем формат.

In [6]:
sample = dataset['validation'][0]
print("Контекст:", sample['context'][:200], "...")
print("\nВопрос:", sample['question'])
print("\nОтветы:", sample['answers'])

Контекст: Первые упоминания о строении человеческого тела встречаются в Древнем Египте. В XXVII веке до н. э. египетский врач Имхотеп описал некоторые органы и их функции, в частности головной мозг, деятельност ...

Вопрос: Где встречаются первые упоминания о строении человеческого тела?

Ответы: {'text': ['в Древнем Египте'], 'answer_start': [60]}


Подготовка выборки из 50 примеров

    Формируем DataFrame для оценки. Берём сплит validation — он меньше и репрезентативнее для тестирования. В качестве эталонного ответа используем первый вариант из списка ответов.

In [7]:
NUM_SAMPLES = 50

data = dataset['validation']
data_subset = data.select(range(min(NUM_SAMPLES, len(data))))

samples = []
for item in data_subset:
  gt_answers = item['answers']['text']
  if gt_answers and len(gt_answers) > 0:
     samples.append({
     'context': item['context'],
     'question': item['question'],
     'ground_truth': gt_answers[0]
     })

df = pd.DataFrame(samples)
print(f"Всего примеров для оценки: {len(df)}")
df.head()


Всего примеров для оценки: 50


,context,question,ground_truth
0,Первые упоминания о строении человеческого тел...,Где встречаются первые упоминания о строении ч...,в Древнем Египте
1,Первые упоминания о строении человеческого тел...,Когда египетский врач Имхотеп впервые описал н...,В XXVII веке до н. э.
2,Телескоп имеет модульную структуру и содержит ...,Как называется корректирующая оптическая систе...,COSTAR
3,Критики теории Вегенера поставили во главу угл...,Какая теория была отвергнута после смерти Веге...,теория дрейфа материков
4,При нагревании кусочки янтаря становятся очень...,Чему не уступают по красоте изделия из прессов...,изделиям из монолитных камней


Шаг 3: Определение устройства вычислений

    Проверяем, доступен ли GPU. Все модели будут загружены на доступное устройство. Если только CPU — генерация 200 примеров тремя моделями будет очень медленной

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

if device.type == 'cpu':
   print("⚠️ GPU не обнаружен. Генерация на CPU может быть очень медленной.")
   print("   Рекомендуется Google Colab с GPU или уменьшение NUM_SAMPLES до 20-50")

Используемое устройство: cpu
⚠️ GPU не обнаружен. Генерация на CPU может быть очень медленной.
   Рекомендуется Google Colab с GPU или уменьшение NUM_SAMPLES до 20-50


Авторизуемся на hugging face

In [9]:
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)


Модель 1: ruGPT-3 Medium

Русскоязычная GPT среднего размера (~400M). Легче и быстрее, чем Large версия. Causal LM — генерирует продолжение текста.

In [10]:
print("Загрузка ruGPT3-Medium...")

rugpt_tokenizer = AutoTokenizer.from_pretrained('ai-forever/rugpt3medium_based_on_gpt2')
rugpt_model = AutoModelForCausalLM.from_pretrained(
    'ai-forever/rugpt3medium_based_on_gpt2',
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map='auto' if device.type == 'cuda' else None
    )

if device.type == 'cpu':
   rugpt_model = rugpt_model.to(device)

rugpt_model.eval()
print("✅ ruGPT3-Medium загружена")

Загрузка ruGPT3-Medium...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3medium_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ruGPT3-Medium загружена


Модель 2: ruGPT-3 Large

Большая русскоязычная GPT (~760M). Более качественная генерация, но медленнее. Тоже causal LM.

In [11]:
print("Загрузка ruGPT3-Large...")

rugpt2_tokenizer = AutoTokenizer.from_pretrained('ai-forever/rugpt3large_based_on_gpt2')
rugpt2_model = AutoModelForCausalLM.from_pretrained(
     'ai-forever/rugpt3large_based_on_gpt2',
      torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
      device_map='auto' if device.type == 'cuda' else None
      )

if device.type == 'cpu':
  rugpt2_model = rugpt2_model.to(device)

rugpt2_model.eval()
print("✅ ruGPT3-Large загружена")

Загрузка ruGPT3-Large...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ruGPT3-Large загружена


Модель 3: ruT5 Large

  Seq2Seq-модель от Сбера. В отличие от GPT — генерирует ответ «с нуля», а не продолжает текст. Архитектура T5 изначально обучалась на QA-задачах, поэтому ожидается лучший Exact Match.

In [14]:
print("Загрузка ruT5-base...")

rut5_tokenizer = AutoTokenizer.from_pretrained('ai-forever/ruT5-base')
rut5_model = AutoModelForSeq2SeqLM.from_pretrained(
      'ai-forever/ruT5-base',
      torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
      device_map='auto' if device.type == 'cuda' else None
      )

if device.type == 'cpu':
  rut5_model = rut5_model.to(device)

rut5_model.eval()
print("✅ ruT5-base загружена")

Загрузка ruT5-base...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ ruT5-base загружена


Функция генерации для ruGPT-3 (causal LM)

  Обе GPT-модели работают одинаково: получают промпт (контекст + вопрос) и продолжают текст. Обрезаем ответ по первой строке/точке, чтобы получить краткий ответ, а не продолжение рассуждений.

In [15]:
def generate_gpt(context, question, model, tokenizer, max_tokens=50):
    prompt = f"Контекст: {context}\n\nВопрос: {question}\n\nОтвет:"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2
        )
    elapsed = time.time() - start_time

     # Декодируем только сгенерированную часть (без промпта)
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    generated = generated.strip().split('\n')[0].strip()

     # Обрезаем слишком длинные ответы
    if len(generated) > 100:
        generated = generated[:100].rsplit(' ', 1)[0]

    return generated, elapsed

 Функция генерации для ruT5 (seq2seq)

  T5 не продолжает текст — генерирует ответ с нуля. Используем beam search (deterministic) для более точных ответов. Формат промпта: question: ... context: ....

In [16]:
def generate_t5(context, question, model, tokenizer, max_tokens=50):
    prompt = f"question: {question} context: {context}"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            num_beams=4,
            do_sample=False,
            repetition_penalty=2.5
        )
    elapsed = time.time() - start_time

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    return generated, elapsed

 Словарь моделей для итерации

  Собираем все модели в словарь. Ключ — название модели для результатов, значение — кортеж (объект модели, токенизатор, функция генерации).

In [17]:

models_config = {
    'ruGPT3-Medium': (rugpt_model, rugpt_tokenizer, generate_gpt),
    'ruGPT3-Large':  (rugpt2_model, rugpt2_tokenizer, generate_gpt),
    'ruT5-Base':     (rut5_model, rut5_tokenizer, generate_t5),
}


Тест на одном примере

Проверяем, что все модели генерируют ответы. Запускаем на первом примере из датасета.

In [18]:
test_idx = 0
test_ctx = df.iloc[test_idx]['context']
test_q = df.iloc[test_idx]['question']
test_gt = df.iloc[test_idx]['ground_truth']

print(f"Вопрос: {test_q}")
print(f"Эталонный ответ: {test_gt}")
print("-" * 60)

for name, (model, tokenizer, gen_fn) in models_config.items():
    try:
        answer, t = gen_fn(test_ctx, test_q, model, tokenizer)
        print(f"{name}: {answer} (время: {t:.2f}с)")
    except Exception as e:
        print(f"{name}: ОШИБКА — {e}")

Вопрос: Где встречаются первые упоминания о строении человеческого тела?
Эталонный ответ: в Древнем Египте
------------------------------------------------------------
ruGPT3-Medium: В Книге Бытия (Исх 18:5). Человек состоит из четырёх частей: головы, туловища, ног и рук; четыре (время: 10.91с)
ruGPT3-Large: Известно лишь одно место на Земле — остров Гаити. Там в 1833 году была найдена человеческая кисть (время: 21.02с)
ruT5-Base: человека., (время: 8.04с)


Генерация ответов на всех 200 примерах

In [19]:
results = []

for model_name, (model, tokenizer, gen_fn) in models_config.items():
    print(f"\n{'='*60}")
    print(f"Генерация моделью: {model_name}")
    print(f"{'='*60}")

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=model_name):
        try:
            prediction, elapsed = gen_fn(row['context'], row['question'], model, tokenizer)
            results.append({
                'model': model_name,
                'question': row['question'],
                'context': row['context'],
                'ground_truth': row['ground_truth'],
                'prediction': prediction,
                'time_seconds': elapsed,
                'prediction_length': len(prediction)
            })
        except Exception as e:
            print(f"\nОшибка на примере {idx}: {e}")
            results.append({
                'model': model_name,
                'question': row['question'],
                'context': row['context'],
                'ground_truth': row['ground_truth'],
                'prediction': '',
                'time_seconds': 0,
                'prediction_length': 0
            })

results_df = pd.DataFrame(results)
print(f"\n✅ Всего сгенерировано ответов: {len(results_df)}")
results_df.head()


Генерация моделью: ruGPT3-Medium


ruGPT3-Medium:   0%|          | 0/50 [00:00<?, ?it/s]


Генерация моделью: ruGPT3-Large


ruGPT3-Large:   0%|          | 0/50 [00:00<?, ?it/s]


Генерация моделью: ruT5-Base


ruT5-Base:   0%|          | 0/50 [00:00<?, ?it/s]


✅ Всего сгенерировано ответов: 150


,model,question,context,ground_truth,prediction,time_seconds,prediction_length
0,ruGPT3-Medium,Где встречаются первые упоминания о строении ч...,Первые упоминания о строении человеческого тел...,в Древнем Египте,Когда люди впервые начали изучать строение чел...,10.176059,90
1,ruGPT3-Medium,Когда египетский врач Имхотеп впервые описал н...,Первые упоминания о строении человеческого тел...,В XXVII веке до н. э.,"Так как он был врачом с именем, то его исследо...",10.080552,98
2,ruGPT3-Medium,Как называется корректирующая оптическая систе...,Телескоп имеет модульную структуру и содержит ...,COSTAR,Система COSTAR была установлена перед самой эк...,10.507617,99
3,ruGPT3-Medium,Какая теория была отвергнута после смерти Веге...,Критики теории Вегенера поставили во главу угл...,теория дрейфа материков,Теория происхождения человека. Согласно этой в...,11.186069,93
4,ruGPT3-Medium,Чему не уступают по красоте изделия из прессов...,При нагревании кусочки янтаря становятся очень...,изделиям из монолитных камней,"Камню! Он намного более твёрдый, чем природный...",11.095633,99


Функции для вычисления QA-метрик

Стандартные метрики для QA-задач (как в SQuAD):
  - Exact Match — точное совпадение нормализованного предсказания с эталоном
  - F1-score — пересечение токенов (precision × recall / 2)
  - BLEU — совпадение n-gram с smoothing для коротких ответов

In [20]:
def normalize_answer(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = ' '.join(text.split())
    return text


def exact_match(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))


def f1_score(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()

    if not gt_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens) if pred_tokens else 0
    recall = num_same / len(gt_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * (precision * recall) / (precision + recall)


def bleu_score(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = [normalize_answer(ground_truth).split()]

    if not pred_tokens or not gt_tokens[0]:
        return 0.0

    smoothing = SmoothingFunction().method1
    return sentence_bleu(gt_tokens, pred_tokens, smoothing_function=smoothing)

Вычисление метрик для каждого ответа

Применяем все три метрики к каждой строке results_df.

In [21]:
results_df['em'] = results_df.apply(
    lambda row: exact_match(row['prediction'], row['ground_truth']), axis=1
)
results_df['f1'] = results_df.apply(
    lambda row: f1_score(row['prediction'], row['ground_truth']), axis=1
)
results_df['bleu'] = results_df.apply(
    lambda row: bleu_score(row['prediction'], row['ground_truth']), axis=1
)

print("✅ Метрики F1, EM, BLEU рассчитаны")
results_df[['model', 'ground_truth', 'prediction', 'em', 'f1', 'bleu']].head(10)

✅ Метрики F1, EM, BLEU рассчитаны


,model,ground_truth,prediction,em,f1,bleu
0,ruGPT3-Medium,в Древнем Египте,Когда люди впервые начали изучать строение чел...,0,0.000000,0.000000
1,ruGPT3-Medium,В XXVII веке до н. э.,"Так как он был врачом с именем, то его исследо...",0,0.000000,0.000000
2,ruGPT3-Medium,COSTAR,Система COSTAR была установлена перед самой эк...,0,0.125000,0.013218
3,ruGPT3-Medium,теория дрейфа материков,Теория происхождения человека. Согласно этой в...,0,0.125000,0.015537
4,ruGPT3-Medium,изделиям из монолитных камней,"Камню! Он намного более твёрдый, чем природный...",0,0.000000,0.000000
5,ruGPT3-Medium,оральные и назальные,Существует пять групп дифтонгов: 1) сложные гл...,0,0.000000,0.000000
6,ruGPT3-Medium,как хиатусы,Во всех случаях есть два варианта перевода сло...,0,0.000000,0.000000
7,ruGPT3-Medium,на предсказуемость швейцарского правового порядка,В долгосрочной перспективе для клиентов будет ...,0,0.000000,0.000000
8,ruGPT3-Medium,деталями соглашения,Все очень просто. В прошлом году банки получил...,0,0.000000,0.000000
9,ruGPT3-Medium,суд первой инстанции,В Бельгии есть три судебных округа (в том числ...,0,0.111111,0.013218


Сводная таблица метрик по моделям

Агрегируем средние значения метрик по каждой модели. Стандартное отклонение для времени и длины ответа показывает стабильность модели.

In [22]:
agg_metrics = results_df.groupby('model').agg({
    'em': 'mean',
    'f1': 'mean',
    'bleu': 'mean',
    'time_seconds': ['mean', 'std'],
    'prediction_length': ['mean', 'std']
}).round(4)

agg_metrics.columns = ['EM (mean)', 'F1 (mean)', 'BLEU (mean)',
                       'Time (mean)', 'Time (std)',
                       'Length (mean)', 'Length (std)']

print("\n📊 Сводная таблица метрик по моделям:")
agg_metrics


📊 Сводная таблица метрик по моделям:


,EM (mean),F1 (mean),BLEU (mean),Time (mean),Time (std),Length (mean),Length (std)
model,,,,,,,
ruGPT3-Large,0.0,0.0405,0.0085,21.1972,2.4756,94.64,7.9866
ruGPT3-Medium,0.0,0.0272,0.0031,11.1275,1.5867,92.68,13.0438
ruT5-Base,0.0,0.0853,0.0154,4.6209,2.7462,35.52,39.7601


Загрузка модели для семантической близости

Используем paraphrase-multilingual-MiniLM — легковесная мультиязычная модель sentence embeddings. Она создаёт векторные представления текстов: семантически близкие тексты будут иметь высокий
cosine similarity.

In [23]:
print("Загрузка модели семантической близости...")
semantic_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print("✅ Модель загружена")

Загрузка модели семантической близости...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Модель загружена


Вычисление Semantic Similarity

Для каждой пары (ground_truth, prediction) вычисляем косинусную близость эмбеддингов. Обрабатываем батчами по 32 для экономии памяти.

In [24]:

def compute_semantic_similarity(texts1, texts2, batch_size=32):
    similarities = []

    for i in tqdm(range(0, len(texts1), batch_size), desc='Semantic Similarity'):
        batch1 = texts1[i:i+batch_size]
        batch2 = texts2[i:i+batch_size]

        emb1 = semantic_model.encode(batch1, convert_to_tensor=True)
        emb2 = semantic_model.encode(batch2, convert_to_tensor=True)

        cos_sim = util.cos_sim(emb1, emb2)
        # Диагональ — пары (gt_i, pred_i)
        similarities.extend(torch.diag(cos_sim).cpu().numpy().tolist())

    return similarities

print("Вычисление Semantic Similarity...")
results_df['semantic_similarity'] = compute_semantic_similarity(
      results_df['ground_truth'].tolist(),
      results_df['prediction'].tolist()
)

print("✅ Semantic Similarity рассчитана")

Вычисление Semantic Similarity...


Semantic Similarity:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Semantic Similarity рассчитана


Добавляем Semantic Similarity в сводную таблицу

Вычисляем среднюю семантическую близость по каждой модели и обновляем итоговую таблицу.

In [25]:
semantic_by_model = results_df.groupby('model')['semantic_similarity'].mean()
agg_metrics['Semantic Sim (mean)'] = semantic_by_model.round(4)

print("\n📊 Обновлённая сводная таблица:")
agg_metrics


📊 Обновлённая сводная таблица:


,EM (mean),F1 (mean),BLEU (mean),Time (mean),Time (std),Length (mean),Length (std),Semantic Sim (mean)
model,,,,,,,,
ruGPT3-Large,0.0,0.0405,0.0085,21.1972,2.4756,94.64,7.9866,0.3255
ruGPT3-Medium,0.0,0.0272,0.0031,11.1275,1.5867,92.68,13.0438,0.3128
ruT5-Base,0.0,0.0853,0.0154,4.6209,2.7462,35.52,39.7601,0.4249
